# 12-0. 단위 상수 수정 (미래 정보 제거)

12시리즈를 처음 만들 때 모델에 넣는 값의 단위를 맞추는 상수 `C = 100 / 표준편차`를 **전체 데이터의 표준편차**로 정했다(`X3_l1`만 C=1). 이러면 각 origin에서 그 이후 데이터가 표준편차에 들어가 미래 정보가 미세하게 섞인다. 이 상수를 **origin마다 그 시점까지의 훈련 구간 표준편차**로 바꾸고 12~14시리즈를 다시 계산했다.

- **바뀐 것**: 모델 입력의 단위 상수만. 적합, 오차 풀(`o' + h <= o`), 구간 계산은 그대로다.
- **점수 정규화 단위**(전체 표준편차)는 모든 조합에 같은 상수라 순위와 선정에 영향이 없어 그대로 둔다.
- **비교 방식**: 단위 상수 효과만 보려고 두 계산 모두 리샘플링 난수 시드를 42로 같게 하고 비교한다. 이전 값은 옛 단위 상수 + 시드 42로 다시 계산한 것(`notebooks/cache/_legacy_scale_seed42/`)이다.
- **시드 고정(4절)**: 이 수정과 별개로 리샘플링 난수 시드를 모든 변수, 모델, (d, D)에서 42 하나로 통일했다. 그 영향은 4절에서 본다(시드를 바꾸기 전 캐시 `notebooks/cache/_before_seed/` 사용).

## 1. 이전·수정 후 결과 읽기

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
KEY = ["모델", "d", "D", "스케일", "구간"]
# 변수: (이름, 이전 캐시, 수정 후 캐시)
VARS = [("X3_l1 (X3)", "cache_1215_X3_l1_df.pkl"), ("X6_l1 (X6)", "cache_127_X6_l1_df.pkl"),
        ("X2 (X2)", "cache_128_X2_df.pkl"), ("X1_l1 (X1)", "cache_129_X1_l1_df.pkl"),
        ("X5_l1 (X5)", "cache_1210_X5_l1_df.pkl"), ("X4 (X4)", "cache_1211_X4_df.pkl")]


def table(r):
    per = r.fillna({"crps": np.inf, "MSE": np.inf, "폭": np.inf}).groupby(KEY + ["origin"])[["crps", "MSE", "적중", "폭"]].mean(skipna=False)
    return per.groupby(KEY).mean(skipna=False).rename(columns={"crps": "CRPS", "적중": "커버리지"})


def pick(t):
    f = t[np.isfinite(t["CRPS"])]
    best = f["CRPS"].min()
    c = f[f["CRPS"] <= best * 1.10]
    c2 = c.assign(g=(c["커버리지"] - 90).abs()).sort_values(["g", "CRPS"])
    return tuple(c2.index[0]), set(c.index), best


def lab(s):
    return f"{s[0]}·({s[1]},{s[2]})·{s[3]}·{s[4]}"


OLD, NEW = {}, {}
for nm, f in VARS:
    OLD[nm] = table(pd.read_pickle("../cache/_legacy_scale_seed42/" + f))
    NEW[nm] = table(pd.read_pickle("../cache/" + f))
print("이전 캐시(전체 표준편차)와 수정 후 캐시(훈련 구간 표준편차)를 읽었다:", len(VARS), "개 변수")

이전 캐시(전체 표준편차)와 수정 후 캐시(훈련 구간 표준편차)를 읽었다: 6 개 변수


## 2. 선정 모델 비교

두 방식의 선정(CRPS 1등의 ±10% 이내 중 커버리지가 90%에 가장 가까운 것)과 조합별 CRPS 변화다.

In [2]:
rows = []
for nm, _ in VARS:
    o, n = OLD[nm], NEW[nm]
    so, co, bo = pick(o)
    sn, cn, bn = pick(n)
    idx = [i for i in o.index if np.isfinite(o["CRPS"][i]) and np.isfinite(n["CRPS"][i])]
    ch = ((n["CRPS"][idx] / o["CRPS"][idx] - 1) * 100).abs()
    rows.append({"변수": nm, "이전 선정": lab(so), "수정 후 선정": lab(sn), "선정 같음": so == sn, "CRPS 1등(이전)": bo, "CRPS 1등(수정 후)": bn,
                 "후보 수(이전 / 수정 후)": f"{len(co)} / {len(cn)}", "후보 집합 같음": co == cn,
                 "조합별 CRPS 변화 평균(%)": ch.mean(), "최대(%)": ch.max()})
CMP = pd.DataFrame(rows).set_index("변수")
with pd.option_context("display.float_format", "{:,.4f}".format, "display.width", 250, "display.max_columns", 20):
    display(CMP)

,이전 선정,수정 후 선정,선정 같음,CRPS 1등(이전),CRPS 1등(수정 후),후보 수(이전 / 수정 후),후보 집합 같음,조합별 CRPS 변화 평균(%),최대(%)
변수,,,,,,,,,
X3_l1 (X3),"GARCH·(1,0)·원·리샘플링","GARCH·(1,0)·원·리샘플링",True,0.5783,0.5783,13 / 13,True,0.3662,2.7707
X6_l1 (X6),"SV·(1,0)·원·자체","SV·(1,0)·원·자체",True,0.2538,0.2538,14 / 14,False,1.0787,32.0558
X2 (X2),"AR단독·(0,0)·원·자체","AR단독·(0,0)·원·자체",True,0.5248,0.5248,2 / 2,True,0.1625,1.1201
X1_l1 (X1),"GARCH·(0,0)·원·리샘플링","GARCH·(0,0)·원·리샘플링",True,0.7285,0.7285,6 / 6,True,0.0013,0.0114
X5_l1 (X5),"GARCH·(0,0)·로그·자체","GARCH·(0,0)·로그·자체",True,0.7116,0.7116,3 / 3,True,0.5988,8.5141
X4 (X4),"GARCH·(0,0)·원·리샘플링","GARCH·(0,0)·원·리샘플링",True,0.5527,0.5527,5 / 5,True,0.3250,1.6030


## 3. 변화가 큰 조합

변수마다 CRPS가 가장 많이 바뀐 조합 2개다. `1등 대비`는 수정 후 CRPS를 수정 후 1등 CRPS로 나눈 값으로, 1.10 이하이면 후보다.

In [3]:
rows = []
for nm, _ in VARS:
    o, n = OLD[nm], NEW[nm]
    idx = [i for i in o.index if np.isfinite(o["CRPS"][i]) and np.isfinite(n["CRPS"][i])]
    d = pd.DataFrame({"이전 CRPS": o["CRPS"][idx], "수정 후 CRPS": n["CRPS"][idx]})
    d["변화(%)"] = (d["수정 후 CRPS"] / d["이전 CRPS"] - 1) * 100
    d["1등 대비(수정 후)"] = d["수정 후 CRPS"] / n["CRPS"][np.isfinite(n["CRPS"])].min()
    top = d.reindex(d["변화(%)"].abs().sort_values(ascending=False).index).head(2)
    for i, r_ in top.iterrows():
        rows.append({"변수": nm, "조합": lab(i), "이전 CRPS": r_["이전 CRPS"], "수정 후 CRPS": r_["수정 후 CRPS"], "변화(%)": r_["변화(%)"], "1등 대비(수정 후)": r_["1등 대비(수정 후)"]})
with pd.option_context("display.float_format", "{:,.3f}".format, "display.width", 250):
    display(pd.DataFrame(rows).set_index(["변수", "조합"]))

이전 CRPS  수정 후 CRPS   변화(%)  1등 대비(수정 후)
변수         조합                                                         
X3_l1 (X3) GARCH·(1,1)·원·자체      0.835      0.812  -2.771        1.405
           AR단독·(0,0)·로그·리샘플링    0.627      0.612  -2.472        1.058
X6_l1 (X6) GARCH·(0,1)·원·리샘플링    0.565      0.384 -32.056        1.513
           GARCH·(1,1)·원·리샘플링    0.295      0.279  -5.347        1.100
X2 (X2)    GARCH·(1,1)·원·리샘플링    1.036      1.024  -1.120        1.952
           GARCH·(1,0)·원·리샘플링    0.645      0.638  -1.062        1.216
X1_l1 (X1) GARCH·(0,0)·원·리샘플링    0.745      0.745  -0.011        1.023
           AR단독·(1,1)·원·리샘플링     1.292      1.292  -0.005        1.773
X5_l1 (X5) GARCH·(0,1)·원·리샘플링    1.656      1.515  -8.514        2.130
           GARCH·(0,1)·원·자체      1.559      1.435  -7.981        2.016
X4 (X4)    AR단독·(1,1)·원·리샘플링     0.829      0.816  -1.603        1.476
           AR단독·(0,1)·원·리샘플링     0.767      0.755  -1.596        1.366

## 4. 리샘플링 시드를 42로 고정한 영향

리샘플링은 오차 풀에서 난수로 500개를 뽑는다. 변수와 모델, (d, D)가 모두 같은 시드(42)를 써서 같은 난수로 비교하도록 통일했다. 점예측, MSE, 자체 구간은 난수를 쓰지 않아 바뀌지 않는다. 아래는 시드를 바꾸기 전 캐시(옛 시드 이름, 새 단위 상수)와 지금 캐시(시드 42, 새 단위 상수)의 비교다. 시드 5개에 따른 선정의 안정성은 `14-5`에서 본다.

In [4]:
BEFORE = {nm: table(pd.read_pickle("../cache/_before_seed/" + f)) for nm, f in VARS}
rows = []
for nm, _ in VARS:
    b_, n_ = BEFORE[nm], NEW[nm]
    sb, cb, bb = pick(b_)
    sn, cn, bn = pick(n_)
    ok = lambda kind: [i for i in b_.index if i[-1] == kind and np.isfinite(b_["CRPS"][i]) and np.isfinite(n_["CRPS"][i])]
    own, rs = ok("자체"), ok("리샘플링")
    rows.append({"변수": nm, "시드 변경 전 선정": lab(sb), "시드 42 선정": lab(sn), "선정 같음": sb == sn,
                 "자체 구간 CRPS 변화(%)": ((n_["CRPS"][own] / b_["CRPS"][own] - 1) * 100).abs().max(),
                 "리샘플링 구간 CRPS 변화 평균(%)": ((n_["CRPS"][rs] / b_["CRPS"][rs] - 1) * 100).abs().mean()})
with pd.option_context("display.float_format", "{:,.2f}".format, "display.width", 250, "display.max_columns", 20):
    display(pd.DataFrame(rows).set_index("변수"))

,시드 변경 전 선정,시드 42 선정,선정 같음,자체 구간 CRPS 변화(%),리샘플링 구간 CRPS 변화 평균(%)
변수,,,,,
X3_l1 (X3),"GARCH·(1,0)·원·리샘플링","GARCH·(1,0)·원·리샘플링",True,0.00,0.70
X6_l1 (X6),"SV·(1,0)·원·자체","SV·(1,0)·원·자체",True,0.00,0.26
X2 (X2),"AR단독·(0,0)·원·자체","AR단독·(0,0)·원·자체",True,0.00,0.50
X1_l1 (X1),"SV·(0,0)·원·리샘플링","GARCH·(0,0)·원·리샘플링",False,0.00,0.56
X5_l1 (X5),"GARCH·(0,0)·로그·자체","GARCH·(0,0)·로그·자체",True,0.00,1.48
X4 (X4),"GARCH·(0,0)·원·리샘플링","GARCH·(0,0)·원·리샘플링",True,0.00,1.02


## 결론

**단위 상수 수정(시드 42로 같게 맞춘 비교)**
- 선정 모델은 6개 변수 모두 같다. CRPS 1등의 값도 모두 같다.
- 후보 집합은 X6_l1만 달라진다(1개가 바뀌고 후보 수는 14개로 같음).
- 조합별 CRPS 변화는 평균 0.00~1.08%이고, 가장 큰 변화는 후보 밖의 조합(X6_l1, -32%)이다. 변화는 (d, D)가 d=1·D=0인 조합에서 작고 d=0·D=1 조합에서 크다.
- 선정된 조합의 점수는 대부분 같다. 달라지는 것은 X3(CRPS 0.6213 -> 0.6228, 커버리지 83.0 -> 84.1%)와 X4(CRPS 0.5931 -> 0.5918, 커버리지 90.9 -> 87.5%)다.

**리샘플링 시드를 42로 고정한 영향**
- 모든 변수, 모델, (d, D)가 같은 시드를 써서 같은 난수로 비교한다. 점예측, MSE, 자체 구간은 난수를 쓰지 않아 바뀌지 않고, 리샘플링 구간의 CRPS가 조합 평균 0.26~1.48% 바뀐다.
- 선정은 X1만 바뀐다(SV·(0,0)·원·리샘플링 -> GARCH·(0,0)·원·리샘플링, CRPS 0.7515 -> 0.7453). 두 후보는 CRPS와 커버리지가 거의 같은 동점이어서 난수에 따라 뒤집힌다(14-5에서 시드 5개로 확인).
- 이 변경으로 12-4~12-13, 13시리즈, 14시리즈, PPT를 다시 만들었다. 12-3(구현 검증)과 12-14, 12-15(smearing)는 로그 스케일 계산이라 영향이 없어 그대로 둔다.
